# Dữ liệu ung thư phổi: khám phá và tiền xử lý

*Bài 4, mục 4.3.1 – 4.3.2.* Bộ dữ liệu **Lung Cancer** trên Kaggle: <https://www.kaggle.com/datasets/mysarahmadbhat/lung-cancer>, file `survey lung cancer.csv` (đặt trong thư mục `../data/`).

In [1]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

DATA = Path('..') / 'data'

## 1. Đọc và khám phá dữ liệu (4.3.1)

In [2]:
df = pd.read_csv(DATA / 'survey lung cancer.csv')
print(df.shape)
df.head()

(309, 16)


,GENDER,AGE,SMOKING,YELLOW_FINGERS,ANXIETY,PEER_PRESSURE,CHRONIC DISEASE,FATIGUE,ALLERGY,WHEEZING,ALCOHOL CONSUMING,COUGHING,SHORTNESS OF BREATH,SWALLOWING DIFFICULTY,CHEST PAIN,LUNG_CANCER
0,M,69,1,2,2,1,1,2,1,2,2,2,2,2,2,YES
1,M,74,2,1,1,1,2,2,2,1,1,1,2,2,2,YES
2,F,59,1,1,1,2,1,2,1,2,1,2,2,1,2,NO
3,M,63,2,2,2,1,1,1,1,1,2,1,1,2,2,NO
4,F,63,1,2,1,1,1,1,1,2,1,2,2,1,1,NO


In [3]:
df.info()
print('Giá trị thiếu:', df.isna().sum().sum())
print(df['LUNG_CANCER'].value_counts())

<class 'pandas.DataFrame'>
RangeIndex: 309 entries, 0 to 308
Data columns (total 16 columns):
 #   Column                 Non-Null Count  Dtype
---  ------                 --------------  -----
 0   GENDER                 309 non-null    str  
 1   AGE                    309 non-null    int64
 2   SMOKING                309 non-null    int64
 3   YELLOW_FINGERS         309 non-null    int64
 4   ANXIETY                309 non-null    int64
 5   PEER_PRESSURE          309 non-null    int64
 6   CHRONIC DISEASE        309 non-null    int64
 7   FATIGUE                309 non-null    int64
 8   ALLERGY                309 non-null    int64
 9   WHEEZING               309 non-null    int64
 10  ALCOHOL CONSUMING      309 non-null    int64
 11  COUGHING               309 non-null    int64
 12  SHORTNESS OF BREATH    309 non-null    int64
 13  SWALLOWING DIFFICULTY  309 non-null    int64
 14  CHEST PAIN             309 non-null    int64
 15  LUNG_CANCER            309 non-null    str  
dtypes

Chú ý tên cột `'FATIGUE '` và `'ALLERGY '` có khoảng trắng thừa ở cuối.

In [4]:
print(df.columns.tolist())

['GENDER', 'AGE', 'SMOKING', 'YELLOW_FINGERS', 'ANXIETY', 'PEER_PRESSURE', 'CHRONIC DISEASE', 'FATIGUE ', 'ALLERGY ', 'WHEEZING', 'ALCOHOL CONSUMING', 'COUGHING', 'SHORTNESS OF BREATH', 'SWALLOWING DIFFICULTY', 'CHEST PAIN', 'LUNG_CANCER']


## 2. Tiền xử lý (4.3.2)

1. Chuẩn hóa tên cột.
2. Xóa dòng trùng lặp **trước khi** chia tập, tránh cùng một mẫu vừa ở tập huấn luyện vừa ở tập kiểm tra.
3. Mã hóa về 0/1: giới tính (nam = 1), nhãn (có bệnh = 1), các cột triệu chứng từ 1/2 thành 0/1.

In [5]:
df.columns = df.columns.str.strip()

print('Số dòng trùng:', df.duplicated().sum())
df = df.drop_duplicates()
print(df.shape)

df['GENDER'] = (df['GENDER'] == 'M').astype(int)             # nam = 1, nữ = 0
df['LUNG_CANCER'] = (df['LUNG_CANCER'] == 'YES').astype(int) # có bệnh = 1 (lớp dương)
symptoms = df.columns.drop(['GENDER', 'AGE', 'LUNG_CANCER'])
df[symptoms] = df[symptoms] - 1                # 1/2 -> 0/1

print(df['LUNG_CANCER'].value_counts())
df.head(3).T

Số dòng trùng: 33
(276, 16)
LUNG_CANCER
1    238
0     38
Name: count, dtype: int64


,0,1,2
GENDER,1,1,0
AGE,69,74,59
SMOKING,0,1,0
YELLOW_FINGERS,1,0,0
ANXIETY,1,0,0
PEER_PRESSURE,0,0,1
CHRONIC DISEASE,0,1,0
FATIGUE,1,1,1
ALLERGY,0,1,0
WHEEZING,1,0,1


Dữ liệu **mất cân bằng mạnh** (khoảng 86% có bệnh), nên khi chia tập dùng `stratify=y` để giữ tỉ lệ hai lớp.

In [6]:
X = df.drop(columns='LUNG_CANCER')
y = df['LUNG_CANCER']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)
print(len(X_train), len(X_test))
print(y_test.value_counts())

220 56
LUNG_CANCER
1    48
0     8
Name: count, dtype: int64
